### Load Packages

In [1]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import os
import math
import matplotlib.colors as mcolors
import matplotlib.tri as tri
import torch
import gpytorch
from pyproj import Proj, transform
from matplotlib import rcParams as rc
import matplotlib
from mpl_toolkits.mplot3d import Axes3D
from sklearn.preprocessing import StandardScaler
from pickle import dump, load
import linear_operator
import random

#import user models
#from pylib_gp_twoscale import VelGPModel, Genevarikernel
from pylib_gp_twoscale import VelGPModel, Genevarikernel

configs = {
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'font.sans-serif': ['Times New Roman'],
     #'font.sans-serif': ['Microsoft YaHei', 'SimHei', 'Arial', 'Liberation Sans', 'sans-serif'],
    #'font.sans-serif': ['Palatino Linotype', 'Microsoft YaHei', 'SimHei', 'Arial', 'Liberation Sans', 'sans-serif'],
    'axes.grid': True,
    'grid.alpha': 0.5,
    'grid.linestyle': '--',
    'mathtext.fontset': "cm",
    # 'mathtext.rm': "asana math",
    # 'mathtext.cal': "asana math",
    "font.size": 16,
    # 'axes.labelsize': 'large',
    # 'axes.labelpad': 8.0,
    # 'xtick.major.pad': 9.0,
}
rc.update(configs)

In [2]:
norm = mcolors.Normalize(vmin=150., vmax=2000.)  
cmap = matplotlib.colormaps.get_cmap("coolwarm")
red_color = cmap(0.9)
blue_color = cmap(0.1)
gray_color = cmap(0.5)

color_maps = {
    "input": gray_color,
    "Expected": red_color,
    "Predicted": blue_color,
}

### Define Variables

In [4]:
#define corss-section
name_crssect = 'Cross Section ' + str(id)
fname_crssect = name_crssect.lower().replace(' ','_')

#cross-section extents
crssect_depth = [0, 500]

#descritization
crssect_npt_xy = 100
crssect_npt_z  = 20
crs_grid = []

#model directory
dir_model = './'

#model column names (input, output)
cn_x = ['vs_depth','Z_2.5','surf_elevation','easting','northing'] #output
cn_y = 'diff'                                                     #input

#define output directories
dir_out = '../Data/Vel_data/'
dir_fig = '../Data/Figure/'

In [5]:
import numpy as np
import pandas as pd
from pyproj import Transformer

# 4 corners in (lat, lon)
corners = [
    (34.744, -119.53),
    (34.744, -116.9),
    (33.49,  -119.53),
    (33.49,  -116.9),
]

# WGS84 -> UTM Zone 11N (EPSG:32611)
to_utm = Transformer.from_crs("EPSG:4326", "EPSG:32611", always_xy=True)
to_ll  = Transformer.from_crs("EPSG:32611", "EPSG:4326", always_xy=True)

# project corners to UTM
xs, ys = [], []
for lat, lon in corners:
    x, y = to_utm.transform(lon, lat)   # always_xy=True => (lon,lat)
    xs.append(x); ys.append(y)

xmin, xmax = min(xs), max(xs)
ymin, ymax = min(ys), max(ys)

# grid spacing (meters)
dx = dy = 500.0

# include boundary: add a tiny epsilon
xg = np.arange(xmin, xmax + 1e-6, dx)
yg = np.arange(ymin, ymax + 1e-6, dy)

XX, YY = np.meshgrid(xg, yg)

df = pd.DataFrame({
    "easting_m":  XX.ravel(),
    "northing_m": YY.ravel(),
    "utm_zone":   11,
    "utm_hemi":   "N",
})

# (optional) back to lat/lon for plotting in lat/lon
lon, lat = to_ll.transform(df["easting_m"].values, df["northing_m"].values)
df["lon"] = lon
df["lat"] = lat

print(df.head())
print("Total points:", len(df))

# save if you want
# df.to_csv("grid_utm_500m.csv", index=False)

       easting_m    northing_m  utm_zone utm_hemi         lon        lat
0  264942.878321  3.705615e+06        11        N -119.529251  33.464226
1  265442.878321  3.705615e+06        11        N -119.523875  33.464336
2  265942.878321  3.705615e+06        11        N -119.518499  33.464445
3  266442.878321  3.705615e+06        11        N -119.513124  33.464555
4  266942.878321  3.705615e+06        11        N -119.507748  33.464664
Total points: 138876


In [6]:
df

,easting_m,northing_m,utm_zone,utm_hemi,lon,lat
0,264942.878321,3.705615e+06,11,N,-119.529251,33.464226
1,265442.878321,3.705615e+06,11,N,-119.523875,33.464336
2,265942.878321,3.705615e+06,11,N,-119.518499,33.464445
3,266442.878321,3.705615e+06,11,N,-119.513124,33.464555
4,266942.878321,3.705615e+06,11,N,-119.507748,33.464664
...,...,...,...,...,...,...
138871,506942.878321,3.847115e+06,11,N,-116.924130,34.766167
138872,507442.878321,3.847115e+06,11,N,-116.918666,34.766164
138873,507942.878321,3.847115e+06,11,N,-116.913203,34.766160
138874,508442.878321,3.847115e+06,11,N,-116.907739,34.766156


In [7]:
depths = np.arange(0.0, 30.0 + 1e-9, 2.5)  # 0, 2.5, ..., 30

# repeat each XY point for each depth
df3d = df.loc[df.index.repeat(len(depths))].copy()
df3d["depth_m"] = np.tile(depths, len(df))

df3d

,easting_m,northing_m,utm_zone,utm_hemi,lon,lat,depth_m
0,264942.878321,3.705615e+06,11,N,-119.529251,33.464226,0.0
0,264942.878321,3.705615e+06,11,N,-119.529251,33.464226,2.5
0,264942.878321,3.705615e+06,11,N,-119.529251,33.464226,5.0
0,264942.878321,3.705615e+06,11,N,-119.529251,33.464226,7.5
0,264942.878321,3.705615e+06,11,N,-119.529251,33.464226,10.0
...,...,...,...,...,...,...,...
138875,508942.878321,3.847115e+06,11,N,-116.902275,34.766152,20.0
138875,508942.878321,3.847115e+06,11,N,-116.902275,34.766152,22.5
138875,508942.878321,3.847115e+06,11,N,-116.902275,34.766152,25.0
138875,508942.878321,3.847115e+06,11,N,-116.902275,34.766152,27.5


In [9]:
df_ucvm = pd.read_csv('output.txt', sep=r'\s+', 
                      header=None,
                      names=['long','lat','vs_depth','site_elevation','vs30',
                             'cvm','cvm_vp','cvm_vs','cvm_dens',
                             'gtl','gtl_vp','gtl_vs','gtl_dens',
                             'alg','vp','vs','dens'])

In [10]:
Z25_data = np.loadtxt('output2.5.txt')  
z_repeated = np.repeat(Z25_data[:, 2], 13)
#z_repeated = Z25_data[:, 2]
df_ucvm['Z_2.5'] = z_repeated

df_ucvm = df_ucvm.rename(columns={'site_elevation': 'surf_elevation'})

df_ucvm["easting"]  = df3d["easting_m"].to_numpy()
df_ucvm["northing"] = df3d["northing_m"].to_numpy()

In [11]:
df_ucvm

,long,lat,vs_depth,surf_elevation,vs30,cvm,cvm_vp,cvm_vs,cvm_dens,gtl,gtl_vp,gtl_vs,gtl_dens,alg,vp,vs,dens,Z_2.5,easting,northing
0,-119.5293,33.4642,0.0,-969.000,180.0,cvms5,5008.181,2715.391,2456.777,none,0.0,0.0,0.0,crust,5008.181,2715.391,2456.777,0.0,264942.878321,3.705615e+06
1,-119.5293,33.4642,2.5,-969.000,180.0,cvms5,5008.180,2715.391,2456.777,none,0.0,0.0,0.0,crust,5008.180,2715.391,2456.777,0.0,264942.878321,3.705615e+06
2,-119.5293,33.4642,5.0,-969.000,180.0,cvms5,5008.179,2715.390,2456.777,none,0.0,0.0,0.0,crust,5008.179,2715.390,2456.777,0.0,264942.878321,3.705615e+06
3,-119.5293,33.4642,7.5,-969.000,180.0,cvms5,5008.179,2715.390,2456.776,none,0.0,0.0,0.0,crust,5008.179,2715.390,2456.776,0.0,264942.878321,3.705615e+06
4,-119.5293,33.4642,10.0,-969.000,180.0,cvms5,5008.178,2715.389,2456.776,none,0.0,0.0,0.0,crust,5008.178,2715.389,2456.776,0.0,264942.878321,3.705615e+06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1805383,-116.9023,34.7662,20.0,1069.259,468.4,cvms5,4237.189,2463.597,2409.531,none,0.0,0.0,0.0,crust,4237.189,2463.597,2409.531,640.0,508942.878321,3.847115e+06
1805384,-116.9023,34.7662,22.5,1069.259,468.4,cvms5,4237.203,2463.600,2409.532,none,0.0,0.0,0.0,crust,4237.203,2463.600,2409.532,640.0,508942.878321,3.847115e+06
1805385,-116.9023,34.7662,25.0,1069.259,468.4,cvms5,4237.217,2463.602,2409.532,none,0.0,0.0,0.0,crust,4237.217,2463.602,2409.532,640.0,508942.878321,3.847115e+06
1805386,-116.9023,34.7662,27.5,1069.259,468.4,cvms5,4237.231,2463.605,2409.532,none,0.0,0.0,0.0,crust,4237.231,2463.605,2409.532,640.0,508942.878321,3.847115e+06


In [13]:
import geopandas as gpd

gdf_basins = gpd.read_file("../Data/Raw_data/Basins/LA_Basins_merged3.shp")

gdf_basins_utm = gdf_basins.to_crs(epsg=32611)


gdf_pts = gpd.GeoDataFrame(
    df_ucvm.copy(),
    geometry=gpd.points_from_xy(df_ucvm["easting"], df_ucvm["northing"]),
    crs="EPSG:32611"  
)


basin_cols = ["OBJECTID", "Basin_Numb", "Basin_Name", "geometry"]
gdf_basin_small = gdf_basins_utm[basin_cols].copy()


joined = gpd.sjoin(
    gdf_pts,
    gdf_basin_small,
    how="left",
    predicate="within" 
)
joined

,long,lat,vs_depth,surf_elevation,vs30,cvm,cvm_vp,cvm_vs,cvm_dens,gtl,...,vs,dens,Z_2.5,easting,northing,geometry,index_right,OBJECTID,Basin_Numb,Basin_Name
0,-119.5293,33.4642,0.0,-969.000,180.0,cvms5,5008.181,2715.391,2456.777,none,...,2715.391,2456.777,0.0,264942.878321,3.705615e+06,POINT (264942.878 3705615.004),NaN,NaN,NaN,NaN
1,-119.5293,33.4642,2.5,-969.000,180.0,cvms5,5008.180,2715.391,2456.777,none,...,2715.391,2456.777,0.0,264942.878321,3.705615e+06,POINT (264942.878 3705615.004),NaN,NaN,NaN,NaN
2,-119.5293,33.4642,5.0,-969.000,180.0,cvms5,5008.179,2715.390,2456.777,none,...,2715.390,2456.777,0.0,264942.878321,3.705615e+06,POINT (264942.878 3705615.004),NaN,NaN,NaN,NaN
3,-119.5293,33.4642,7.5,-969.000,180.0,cvms5,5008.179,2715.390,2456.776,none,...,2715.390,2456.776,0.0,264942.878321,3.705615e+06,POINT (264942.878 3705615.004),NaN,NaN,NaN,NaN
4,-119.5293,33.4642,10.0,-969.000,180.0,cvms5,5008.178,2715.389,2456.776,none,...,2715.389,2456.776,0.0,264942.878321,3.705615e+06,POINT (264942.878 3705615.004),NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1805383,-116.9023,34.7662,20.0,1069.259,468.4,cvms5,4237.189,2463.597,2409.531,none,...,2463.597,2409.531,640.0,508942.878321,3.847115e+06,POINT (508942.878 3847115.004),NaN,NaN,NaN,NaN
1805384,-116.9023,34.7662,22.5,1069.259,468.4,cvms5,4237.203,2463.600,2409.532,none,...,2463.600,2409.532,640.0,508942.878321,3.847115e+06,POINT (508942.878 3847115.004),NaN,NaN,NaN,NaN
1805385,-116.9023,34.7662,25.0,1069.259,468.4,cvms5,4237.217,2463.602,2409.532,none,...,2463.602,2409.532,640.0,508942.878321,3.847115e+06,POINT (508942.878 3847115.004),NaN,NaN,NaN,NaN
1805386,-116.9023,34.7662,27.5,1069.259,468.4,cvms5,4237.231,2463.605,2409.532,none,...,2463.605,2409.532,640.0,508942.878321,3.847115e+06,POINT (508942.878 3847115.004),NaN,NaN,NaN,NaN


In [14]:
df_ucvm['basin_idnum'] = joined['OBJECTID']
df_ucvm["basin_idnum"] = df_ucvm["basin_idnum"].fillna(1000)
df_ucvm

,long,lat,vs_depth,surf_elevation,vs30,cvm,cvm_vp,cvm_vs,cvm_dens,gtl,...,gtl_vs,gtl_dens,alg,vp,vs,dens,Z_2.5,easting,northing,basin_idnum
0,-119.5293,33.4642,0.0,-969.000,180.0,cvms5,5008.181,2715.391,2456.777,none,...,0.0,0.0,crust,5008.181,2715.391,2456.777,0.0,264942.878321,3.705615e+06,1000.0
1,-119.5293,33.4642,2.5,-969.000,180.0,cvms5,5008.180,2715.391,2456.777,none,...,0.0,0.0,crust,5008.180,2715.391,2456.777,0.0,264942.878321,3.705615e+06,1000.0
2,-119.5293,33.4642,5.0,-969.000,180.0,cvms5,5008.179,2715.390,2456.777,none,...,0.0,0.0,crust,5008.179,2715.390,2456.777,0.0,264942.878321,3.705615e+06,1000.0
3,-119.5293,33.4642,7.5,-969.000,180.0,cvms5,5008.179,2715.390,2456.776,none,...,0.0,0.0,crust,5008.179,2715.390,2456.776,0.0,264942.878321,3.705615e+06,1000.0
4,-119.5293,33.4642,10.0,-969.000,180.0,cvms5,5008.178,2715.389,2456.776,none,...,0.0,0.0,crust,5008.178,2715.389,2456.776,0.0,264942.878321,3.705615e+06,1000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1805383,-116.9023,34.7662,20.0,1069.259,468.4,cvms5,4237.189,2463.597,2409.531,none,...,0.0,0.0,crust,4237.189,2463.597,2409.531,640.0,508942.878321,3.847115e+06,1000.0
1805384,-116.9023,34.7662,22.5,1069.259,468.4,cvms5,4237.203,2463.600,2409.532,none,...,0.0,0.0,crust,4237.203,2463.600,2409.532,640.0,508942.878321,3.847115e+06,1000.0
1805385,-116.9023,34.7662,25.0,1069.259,468.4,cvms5,4237.217,2463.602,2409.532,none,...,0.0,0.0,crust,4237.217,2463.602,2409.532,640.0,508942.878321,3.847115e+06,1000.0
1805386,-116.9023,34.7662,27.5,1069.259,468.4,cvms5,4237.231,2463.605,2409.532,none,...,0.0,0.0,crust,4237.231,2463.605,2409.532,640.0,508942.878321,3.847115e+06,1000.0


In [16]:
df_measure = pd.read_csv(dir_out+'Training_data/df_measure.txt', encoding='utf-8')
df_measure = df_measure.rename(columns={'metadata_id': 'velocity_metadata_id', 'z_2.5': 'Z_2.5',
                                        'site_northing': 'northing', 'site_easting': 'easting',
                                       'vs_middle_depth': 'vs_depth'})
df_measure = df_measure[['vs_depth','Z_2.5','surf_elevation','easting','northing','diff',  'velocity_metadata_id','vs30']]

df_measure

,vs_depth,Z_2.5,surf_elevation,easting,northing,diff,velocity_metadata_id,vs30
0,1.00,2380.0,251.052,352134.046903,3.786794e+06,-1.800524,610,295.692
1,2.00,2380.0,251.052,352134.046903,3.786794e+06,-1.707914,610,295.692
2,3.00,2380.0,251.052,352134.046903,3.786794e+06,-1.565163,610,295.692
3,4.00,2380.0,251.052,352134.046903,3.786794e+06,-1.589788,610,295.692
4,5.00,2380.0,251.052,352134.046903,3.786794e+06,-1.539431,610,295.692
...,...,...,...,...,...,...,...,...
11026,12.60,3620.0,54.938,400420.745906,3.762835e+06,-0.927395,2479,304.365
11027,21.15,3620.0,54.938,400420.745906,3.762835e+06,-0.842869,2479,304.365
11028,34.80,3620.0,54.938,400420.745906,3.762835e+06,-0.710523,2479,304.365
11029,56.60,3620.0,54.938,400420.745906,3.762835e+06,-0.528584,2479,304.365


In [17]:
gdf_basins = gpd.read_file("../Data/Raw_data/Basins/LA_Basins_merged3.shp")

gdf_basins_utm = gdf_basins.to_crs(epsg=32611)


gdf_pts = gpd.GeoDataFrame(
    df_measure.copy(),
    geometry=gpd.points_from_xy(df_measure["easting"], df_measure["northing"]),
    crs="EPSG:32611"  
)


basin_cols = ["OBJECTID", "Basin_Numb", "Basin_Name", "geometry"]
gdf_basin_small = gdf_basins_utm[basin_cols].copy()


joined = gpd.sjoin(
    gdf_pts,
    gdf_basin_small,
    how="left",
    predicate="within" 
)
joined

,vs_depth,Z_2.5,surf_elevation,easting,northing,diff,velocity_metadata_id,vs30,geometry,index_right,OBJECTID,Basin_Numb,Basin_Name
0,1.00,2380.0,251.052,352134.046903,3.786794e+06,-1.800524,610,295.692,POINT (352134.047 3786793.966),1.0,2.0,4-012,SAN FERNANDO VALLEY
1,2.00,2380.0,251.052,352134.046903,3.786794e+06,-1.707914,610,295.692,POINT (352134.047 3786793.966),1.0,2.0,4-012,SAN FERNANDO VALLEY
2,3.00,2380.0,251.052,352134.046903,3.786794e+06,-1.565163,610,295.692,POINT (352134.047 3786793.966),1.0,2.0,4-012,SAN FERNANDO VALLEY
3,4.00,2380.0,251.052,352134.046903,3.786794e+06,-1.589788,610,295.692,POINT (352134.047 3786793.966),1.0,2.0,4-012,SAN FERNANDO VALLEY
4,5.00,2380.0,251.052,352134.046903,3.786794e+06,-1.539431,610,295.692,POINT (352134.047 3786793.966),1.0,2.0,4-012,SAN FERNANDO VALLEY
...,...,...,...,...,...,...,...,...,...,...,...,...,...
11026,12.60,3620.0,54.938,400420.745906,3.762835e+06,-0.927395,2479,304.365,POINT (400420.746 3762835.246),4.0,1.0,4-011,COASTAL PLAIN OF LOS ANGELES
11027,21.15,3620.0,54.938,400420.745906,3.762835e+06,-0.842869,2479,304.365,POINT (400420.746 3762835.246),4.0,1.0,4-011,COASTAL PLAIN OF LOS ANGELES
11028,34.80,3620.0,54.938,400420.745906,3.762835e+06,-0.710523,2479,304.365,POINT (400420.746 3762835.246),4.0,1.0,4-011,COASTAL PLAIN OF LOS ANGELES
11029,56.60,3620.0,54.938,400420.745906,3.762835e+06,-0.528584,2479,304.365,POINT (400420.746 3762835.246),4.0,1.0,4-011,COASTAL PLAIN OF LOS ANGELES


In [18]:
df_measure['basin_idnum'] = joined['OBJECTID']
df_measure["basin_idnum"] = df_measure["basin_idnum"].fillna(1000)
df_measure

,vs_depth,Z_2.5,surf_elevation,easting,northing,diff,velocity_metadata_id,vs30,basin_idnum
0,1.00,2380.0,251.052,352134.046903,3.786794e+06,-1.800524,610,295.692,2.0
1,2.00,2380.0,251.052,352134.046903,3.786794e+06,-1.707914,610,295.692,2.0
2,3.00,2380.0,251.052,352134.046903,3.786794e+06,-1.565163,610,295.692,2.0
3,4.00,2380.0,251.052,352134.046903,3.786794e+06,-1.589788,610,295.692,2.0
4,5.00,2380.0,251.052,352134.046903,3.786794e+06,-1.539431,610,295.692,2.0
...,...,...,...,...,...,...,...,...,...
11026,12.60,3620.0,54.938,400420.745906,3.762835e+06,-0.927395,2479,304.365,1.0
11027,21.15,3620.0,54.938,400420.745906,3.762835e+06,-0.842869,2479,304.365,1.0
11028,34.80,3620.0,54.938,400420.745906,3.762835e+06,-0.710523,2479,304.365,1.0
11029,56.60,3620.0,54.938,400420.745906,3.762835e+06,-0.528584,2479,304.365,1.0


In [19]:
df_measure = df_measure[df_measure['basin_idnum'] != 1000].reset_index(drop=True)
df_measure

,vs_depth,Z_2.5,surf_elevation,easting,northing,diff,velocity_metadata_id,vs30,basin_idnum
0,1.00,2380.0,251.052,352134.046903,3.786794e+06,-1.800524,610,295.692,2.0
1,2.00,2380.0,251.052,352134.046903,3.786794e+06,-1.707914,610,295.692,2.0
2,3.00,2380.0,251.052,352134.046903,3.786794e+06,-1.565163,610,295.692,2.0
3,4.00,2380.0,251.052,352134.046903,3.786794e+06,-1.589788,610,295.692,2.0
4,5.00,2380.0,251.052,352134.046903,3.786794e+06,-1.539431,610,295.692,2.0
...,...,...,...,...,...,...,...,...,...
8899,12.60,3620.0,54.938,400420.745906,3.762835e+06,-0.927395,2479,304.365,1.0
8900,21.15,3620.0,54.938,400420.745906,3.762835e+06,-0.842869,2479,304.365,1.0
8901,34.80,3620.0,54.938,400420.745906,3.762835e+06,-0.710523,2479,304.365,1.0
8902,56.60,3620.0,54.938,400420.745906,3.762835e+06,-0.528584,2479,304.365,1.0


In [20]:
df_sonic = pd.read_csv(dir_out+'Training_data/df_sonic.txt', encoding='utf-8')
df_sonic = df_sonic.rename(columns={'X': 'easting', 'Y': 'northing',
                                       'depth': 'vs_depth'})
df_sonic = df_sonic[['vs_depth','Z_2.5','surf_elevation','easting','northing','diff',  'velocity_metadata_id']]

df_sonic['basin_idnum'] = 1.0
df_sonic

,vs_depth,Z_2.5,surf_elevation,easting,northing,diff,velocity_metadata_id,basin_idnum
0,67.124000,4400.0,23.596,387402.0000,3753149.500,0.010634,-29,1.0
1,77.124000,4400.0,23.596,387402.0000,3753149.500,0.018006,-29,1.0
2,87.124000,4400.0,23.596,387402.0000,3753149.500,0.028627,-29,1.0
3,97.124000,4400.0,23.596,387402.0000,3753149.500,0.040313,-29,1.0
4,107.124000,4400.0,23.596,387402.0000,3753149.500,0.051009,-29,1.0
...,...,...,...,...,...,...,...,...
3177,1956.730197,2500.0,83.353,387298.0938,3767711.984,-0.116894,-28,1.0
3178,1966.730197,2500.0,83.353,387298.0938,3767711.984,-0.056673,-28,1.0
3179,1976.730197,2500.0,83.353,387298.0938,3767711.984,-0.040450,-28,1.0
3180,1986.730197,2500.0,83.353,387298.0938,3767711.984,-0.050204,-28,1.0


In [21]:
df_veldata = pd.concat([df_measure.reset_index(drop=True), df_sonic.reset_index(drop=True)], ignore_index=True, sort=False)
df_veldata

,vs_depth,Z_2.5,surf_elevation,easting,northing,diff,velocity_metadata_id,vs30,basin_idnum
0,1.000000,2380.0,251.052,352134.046903,3.786794e+06,-1.800524,610,295.692,2.0
1,2.000000,2380.0,251.052,352134.046903,3.786794e+06,-1.707914,610,295.692,2.0
2,3.000000,2380.0,251.052,352134.046903,3.786794e+06,-1.565163,610,295.692,2.0
3,4.000000,2380.0,251.052,352134.046903,3.786794e+06,-1.589788,610,295.692,2.0
4,5.000000,2380.0,251.052,352134.046903,3.786794e+06,-1.539431,610,295.692,2.0
...,...,...,...,...,...,...,...,...,...
12081,1956.730197,2500.0,83.353,387298.093800,3.767712e+06,-0.116894,-28,NaN,1.0
12082,1966.730197,2500.0,83.353,387298.093800,3.767712e+06,-0.056673,-28,NaN,1.0
12083,1976.730197,2500.0,83.353,387298.093800,3.767712e+06,-0.040450,-28,NaN,1.0
12084,1986.730197,2500.0,83.353,387298.093800,3.767712e+06,-0.050204,-28,NaN,1.0


### Load and Define GP Model

In [22]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#read model
scaler_input = load(open(dir_model+'ucvm_la_basin_adj_scaler_frozen.pkl', 'rb'))
model_state = torch.load(dir_model+'ucvm_la_basin_adj_model_frozen_newloss.pth', weights_only=True)

#get training input/output
X_train = scaler_input.transform(df_veldata.loc[:,cn_x].values)
Y_train = df_veldata.loc[:,cn_y].values
#convert to tensors
X_train = torch.tensor(X_train, dtype=torch.float64)
Y_train = torch.tensor(Y_train, dtype=torch.float64)



In [23]:
dtype = torch.float32


basin_col = torch.tensor(
    df_veldata['basin_idnum'].to_numpy(),
    dtype=X_train.dtype,
    device=X_train.device
).unsqueeze(1)

X_train = torch.cat([X_train, basin_col], dim=1)
X_train

tensor([[-0.8257, -0.1854,  0.2379, -1.0547,  0.5051,  2.0000],
        [-0.8097, -0.1854,  0.2379, -1.0547,  0.5051,  2.0000],
        [-0.7936, -0.1854,  0.2379, -1.0547,  0.5051,  2.0000],
        ...,
        [30.8894, -0.1001, -0.6438,  0.1647, -0.1566,  1.0000],
        [31.0499, -0.1001, -0.6438,  0.1647, -0.1566,  1.0000],
        [31.2105, -0.1001, -0.6438,  0.1647, -0.1566,  1.0000]],
       dtype=torch.float64)

In [24]:
#define likelihood and model
likelihood = gpytorch.likelihoods.GaussianLikelihood().to(device).double()
model = VelGPModel(X_train, Y_train, likelihood).to(device).double()
model.load_state_dict(model_state)

<All keys matched successfully>

In [25]:
X_train_cross = torch.tensor(scaler_input.transform(df_ucvm.loc[:,cn_x].values), dtype=torch.float64)

basin_col = torch.tensor(
    df_ucvm['basin_idnum'].to_numpy(),
    dtype=X_train.dtype,
    device=X_train.device
).unsqueeze(1)

X_train_cross = torch.cat([X_train_cross, basin_col], dim=1)


Y_train_cross = df_ucvm['vs'].values
Y_train_cross



array([2715.391, 2715.391, 2715.39 , ..., 2463.602, 2463.605, 2463.607])

In [26]:

model.eval()
likelihood.eval()

X = X_train_cross.to(device).float()   
batch_size = 2048 

means = []


with torch.no_grad(), gpytorch.settings.fast_pred_var():
    for i in range(0, X.size(0), batch_size):
        xb = X[i:i+batch_size]
        pred = likelihood(model(xb))
        means.append(pred.mean.detach().cpu())
       

predict_value = torch.cat(means, dim=0)   

In [27]:
df_ucvm['predict_diff'] = predict_value
#df_ucvm['predict_vs'] = df_ucvm["cvm_vs"].to_numpy() * np.exp(df_ucvm["predict_diff"].to_numpy()*df_ucvm['factor'])
df_ucvm['predict_vs'] = df_ucvm["vs"].to_numpy() * np.exp(df_ucvm["predict_diff"].to_numpy())

In [28]:
#df_ucvm["latitude"] = df_ucvm["latitude"].astype(float).round(3)
df_ucvm

,long,lat,vs_depth,surf_elevation,vs30,cvm,cvm_vp,cvm_vs,cvm_dens,gtl,...,alg,vp,vs,dens,Z_2.5,easting,northing,basin_idnum,predict_diff,predict_vs
0,-119.5293,33.4642,0.0,-969.000,180.0,cvms5,5008.181,2715.391,2456.777,none,...,crust,5008.181,2715.391,2456.777,0.0,264942.878321,3.705615e+06,1000.0,0.0,2715.391
1,-119.5293,33.4642,2.5,-969.000,180.0,cvms5,5008.180,2715.391,2456.777,none,...,crust,5008.180,2715.391,2456.777,0.0,264942.878321,3.705615e+06,1000.0,0.0,2715.391
2,-119.5293,33.4642,5.0,-969.000,180.0,cvms5,5008.179,2715.390,2456.777,none,...,crust,5008.179,2715.390,2456.777,0.0,264942.878321,3.705615e+06,1000.0,0.0,2715.390
3,-119.5293,33.4642,7.5,-969.000,180.0,cvms5,5008.179,2715.390,2456.776,none,...,crust,5008.179,2715.390,2456.776,0.0,264942.878321,3.705615e+06,1000.0,0.0,2715.390
4,-119.5293,33.4642,10.0,-969.000,180.0,cvms5,5008.178,2715.389,2456.776,none,...,crust,5008.178,2715.389,2456.776,0.0,264942.878321,3.705615e+06,1000.0,0.0,2715.389
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1805383,-116.9023,34.7662,20.0,1069.259,468.4,cvms5,4237.189,2463.597,2409.531,none,...,crust,4237.189,2463.597,2409.531,640.0,508942.878321,3.847115e+06,1000.0,0.0,2463.597
1805384,-116.9023,34.7662,22.5,1069.259,468.4,cvms5,4237.203,2463.600,2409.532,none,...,crust,4237.203,2463.600,2409.532,640.0,508942.878321,3.847115e+06,1000.0,0.0,2463.600
1805385,-116.9023,34.7662,25.0,1069.259,468.4,cvms5,4237.217,2463.602,2409.532,none,...,crust,4237.217,2463.602,2409.532,640.0,508942.878321,3.847115e+06,1000.0,0.0,2463.602
1805386,-116.9023,34.7662,27.5,1069.259,468.4,cvms5,4237.231,2463.605,2409.532,none,...,crust,4237.231,2463.605,2409.532,640.0,508942.878321,3.847115e+06,1000.0,0.0,2463.605


In [29]:
df_ucvm.to_csv('df_ucvm.txt', index=False,
                  encoding='utf-8',
                  lineterminator='\n')

In [30]:
df_ucvm = pd.read_csv('df_ucvm.txt', encoding='utf-8')
df_ucvm

,long,lat,vs_depth,surf_elevation,vs30,cvm,cvm_vp,cvm_vs,cvm_dens,gtl,...,alg,vp,vs,dens,Z_2.5,easting,northing,basin_idnum,predict_diff,predict_vs
0,-119.5293,33.4642,0.0,-969.000,180.0,cvms5,5008.181,2715.391,2456.777,none,...,crust,5008.181,2715.391,2456.777,0.0,264942.878321,3.705615e+06,1000.0,0.0,2715.391
1,-119.5293,33.4642,2.5,-969.000,180.0,cvms5,5008.180,2715.391,2456.777,none,...,crust,5008.180,2715.391,2456.777,0.0,264942.878321,3.705615e+06,1000.0,0.0,2715.391
2,-119.5293,33.4642,5.0,-969.000,180.0,cvms5,5008.179,2715.390,2456.777,none,...,crust,5008.179,2715.390,2456.777,0.0,264942.878321,3.705615e+06,1000.0,0.0,2715.390
3,-119.5293,33.4642,7.5,-969.000,180.0,cvms5,5008.179,2715.390,2456.776,none,...,crust,5008.179,2715.390,2456.776,0.0,264942.878321,3.705615e+06,1000.0,0.0,2715.390
4,-119.5293,33.4642,10.0,-969.000,180.0,cvms5,5008.178,2715.389,2456.776,none,...,crust,5008.178,2715.389,2456.776,0.0,264942.878321,3.705615e+06,1000.0,0.0,2715.389
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1805383,-116.9023,34.7662,20.0,1069.259,468.4,cvms5,4237.189,2463.597,2409.531,none,...,crust,4237.189,2463.597,2409.531,640.0,508942.878321,3.847115e+06,1000.0,0.0,2463.597
1805384,-116.9023,34.7662,22.5,1069.259,468.4,cvms5,4237.203,2463.600,2409.532,none,...,crust,4237.203,2463.600,2409.532,640.0,508942.878321,3.847115e+06,1000.0,0.0,2463.600
1805385,-116.9023,34.7662,25.0,1069.259,468.4,cvms5,4237.217,2463.602,2409.532,none,...,crust,4237.217,2463.602,2409.532,640.0,508942.878321,3.847115e+06,1000.0,0.0,2463.602
1805386,-116.9023,34.7662,27.5,1069.259,468.4,cvms5,4237.231,2463.605,2409.532,none,...,crust,4237.231,2463.605,2409.532,640.0,508942.878321,3.847115e+06,1000.0,0.0,2463.605


In [31]:
n_per_cell = 13
df_ucvm = df_ucvm.reset_index(drop=True)  
df_ucvm["cell_id"] = (np.arange(len(df_ucvm)) // n_per_cell) + 1
df_ucvm

,long,lat,vs_depth,surf_elevation,vs30,cvm,cvm_vp,cvm_vs,cvm_dens,gtl,...,vp,vs,dens,Z_2.5,easting,northing,basin_idnum,predict_diff,predict_vs,cell_id
0,-119.5293,33.4642,0.0,-969.000,180.0,cvms5,5008.181,2715.391,2456.777,none,...,5008.181,2715.391,2456.777,0.0,264942.878321,3.705615e+06,1000.0,0.0,2715.391,1
1,-119.5293,33.4642,2.5,-969.000,180.0,cvms5,5008.180,2715.391,2456.777,none,...,5008.180,2715.391,2456.777,0.0,264942.878321,3.705615e+06,1000.0,0.0,2715.391,1
2,-119.5293,33.4642,5.0,-969.000,180.0,cvms5,5008.179,2715.390,2456.777,none,...,5008.179,2715.390,2456.777,0.0,264942.878321,3.705615e+06,1000.0,0.0,2715.390,1
3,-119.5293,33.4642,7.5,-969.000,180.0,cvms5,5008.179,2715.390,2456.776,none,...,5008.179,2715.390,2456.776,0.0,264942.878321,3.705615e+06,1000.0,0.0,2715.390,1
4,-119.5293,33.4642,10.0,-969.000,180.0,cvms5,5008.178,2715.389,2456.776,none,...,5008.178,2715.389,2456.776,0.0,264942.878321,3.705615e+06,1000.0,0.0,2715.389,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1805383,-116.9023,34.7662,20.0,1069.259,468.4,cvms5,4237.189,2463.597,2409.531,none,...,4237.189,2463.597,2409.531,640.0,508942.878321,3.847115e+06,1000.0,0.0,2463.597,138876
1805384,-116.9023,34.7662,22.5,1069.259,468.4,cvms5,4237.203,2463.600,2409.532,none,...,4237.203,2463.600,2409.532,640.0,508942.878321,3.847115e+06,1000.0,0.0,2463.600,138876
1805385,-116.9023,34.7662,25.0,1069.259,468.4,cvms5,4237.217,2463.602,2409.532,none,...,4237.217,2463.602,2409.532,640.0,508942.878321,3.847115e+06,1000.0,0.0,2463.602,138876
1805386,-116.9023,34.7662,27.5,1069.259,468.4,cvms5,4237.231,2463.605,2409.532,none,...,4237.231,2463.605,2409.532,640.0,508942.878321,3.847115e+06,1000.0,0.0,2463.605,138876


In [32]:


def vs30_from_group(g, vs_col="predict_vs"):
    gg = g.sort_values("vs_depth")


    gg = gg[gg["vs_depth"].isin([0.0, 2.5, 5.0, 7.5, 10.0, 12.5, 15.0, 17.5,  20.0, 22.5 ,25.0, 27.5 ,30.0])].copy()
    if gg["vs_depth"].nunique() != 13:
        return np.nan


    gg = gg.set_index("vs_depth").loc[[0.0, 2.5, 5.0, 7.5, 10.0, 12.5, 15.0, 17.5,  20.0, 22.5 ,25.0, 27.5, 30.0]]

    z = gg.index.to_numpy(float)
    vs = gg[vs_col].to_numpy(float)

  
    return 30.0 / np.trapz(1.0 / vs, z)


vs30_df = (
    df_ucvm.groupby("cell_id", as_index=False)
          .apply(lambda g: vs30_from_group(g, vs_col="predict_vs"))
          .rename(columns={None: "vs30_pred"})
)

meta = df_ucvm.groupby("cell_id", as_index=False)[
    ["easting", "northing", "long", "lat"]
].first()

vs30_df = vs30_df.merge(meta, on="cell_id", how="left")
vs30_df

/tmp/ipykernel_17548/138694113.py:21: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: vs30_from_group(g, vs_col="predict_vs"))


,cell_id,vs30_pred,easting,northing,long,lat
0,1,2715.387792,264942.878321,3.705615e+06,-119.5293,33.4642
1,2,2717.219625,265442.878321,3.705615e+06,-119.5239,33.4643
2,3,2718.503958,265942.878321,3.705615e+06,-119.5185,33.4644
3,4,2719.836542,266442.878321,3.705615e+06,-119.5131,33.4646
4,5,2722.482083,266942.878321,3.705615e+06,-119.5077,33.4647
...,...,...,...,...,...,...
138871,138872,2436.486375,506942.878321,3.847115e+06,-116.9241,34.7662
138872,138873,2441.125625,507442.878321,3.847115e+06,-116.9187,34.7662
138873,138874,2446.035250,507942.878321,3.847115e+06,-116.9132,34.7662
138874,138875,2453.740292,508442.878321,3.847115e+06,-116.9077,34.7662


In [29]:
vs30_df.to_csv('vs30_df.txt', index=False,
                  encoding='utf-8',
                  lineterminator='\n')